# What do coordinates and text contribute?

This notebook follows one controlled question: **do geological descriptions add useful information beyond XYZ and depth?**

The benchmark crosses 14 sentence-embedding models, several PCA dimensions, and five uncertainty strategies. LOBO holds out one borehole. LOCO holds out an entire survey campaign and therefore also changes writing style and acquisition context.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import bovino_results as br

## 1. Controlled feature sets

We compare depth alone, text with depth, XYZ with depth, and the complete input. For the complete input, the text is represented by 8, 16, 32, or 64 PCA components or by the native embedding.

Every configuration uses the same target definition and split protocol. The benchmark varies one modelling choice at a time so that differences remain interpretable.

In [ ]:
lobo_summary = br.representation_summary('lobo')
loco_summary = br.representation_summary('loco')

display(lobo_summary[['configuration', 'accuracy', 'kappa']].round(3))
display(loco_summary[['configuration', 'accuracy', 'kappa']].round(3))

## 2. The full benchmark

Each line averages a method over the available embedding models. Depth alone performs poorly. Text alone improves on depth but remains below the spatial baseline. Combining text with XYZ and depth gives the strongest overall results.

In [ ]:
fig, axes = br.plot_feature_representation_comparison()
plt.show()

## 3. The validation protocol changes the conclusion

Across the five selected methods, XYZ plus depth reaches about **0.716 accuracy in LOBO** and **0.633 in LOCO**. Adding PCA16 text embeddings raises these averages to about **0.753** and **0.648**, respectively.

Native embeddings remain competitive in LOBO but fall to about **0.590 in LOCO**. Compression therefore matters most when the test campaign has an unseen descriptive style.

In [ ]:
fig, axis = br.plot_lobo_loco_transfer()
plt.show()

## 4. MiniLM as a concrete operating point

MiniLM PCA16 is not claimed to be universally optimal. It is compact, reproducible, and consistently useful in these experiments. For MiniLM, adding PCA16 text to XYZ and depth improves every selected method under both protocols.

In [ ]:
for protocol in ['lobo', 'loco']:
    frame = br.load_representation_metrics(protocol)
    selected = frame[
        (frame['embedding'] == 'minilm-en')
        & frame['uq_method'].isin(br.UQ_METHODS)
        & frame['configuration'].isin(['XYZ + depth', 'Embeddings + XYZ + depth: PCA 16'])
    ]
    table = selected.groupby(['configuration', 'uq_method'])['accuracy'].mean().unstack()
    print(protocol.upper())
    display(table.round(3))

## Take-home message

The descriptions contain predictive information beyond coordinates. The gain is systematic in LOBO and smaller under unseen campaigns. PCA acts as useful regularization under the stronger shift, while uncompressed embeddings can transfer poorly.

The many benchmark cells share the same observations and are not independent statistical replicates. Their agreement supports a robust descriptive conclusion, not a conventional significance test.